# От аналитической таблицы к задаче машинного обучения

Машинное обучение рассматривается как проверяемая процедура получения правила из примеров. Сквозная задача модуля — оценить риск задержки доставки в момент оформления заказа. Положительный класс означает задержку, а не хороший исход.

Исходные таблицы курса используются для проверки постановки и как источник названий и шкал дополнительного учебного набора. Синтетический набор не описывает качество работы реальных городов, покупателей или служб доставки. Все последующие ноутбуки используют один файл и одинаковое временное разбиение. Ноутбук автономен и содержит генерацию этого небольшого дополнительного источника.

## 1. Постановка задачи и возможности исходного набора

Единица наблюдения — отдельная заявка на доставку. Прогноз относится к этой заявке и делается при оформлении. Покупатель может встретиться несколько раз: проверяется работа на будущих заявках существующей клиентской базы, а не перенос на полностью неизвестных покупателей.

**Среда и доступные источники**

Все импорты собраны в первой ячейке. Версии сохраняются в выводе, поскольку состав и представление признаков зависят от библиотек. Исходный `course/data` ищется по известному файлу заказов, а дополнительный набор хранится рядом с ноутбуками модуля.

In [1]:
# Шаг М2.01.01 — Среда и доступные источники
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

MODULE_DIR = Path.cwd()
COURSE_DATA = next((p / 'data' for p in (MODULE_DIR, *MODULE_DIR.parents)
                    if (p / 'data' / 'orders.csv').is_file()), None)
if COURSE_DATA is None:
    raise FileNotFoundError('Нужен исходный каталог course/data')
DATASET_PATH = MODULE_DIR / 'data' / 'delivery_risk.csv'
display(pd.Series({'Python-библиотека Scikit-learn': sklearn.__version__,
                   'Pandas': pd.__version__, 'NumPy': np.__version__,
                   'Исходные таблицы': str(COURSE_DATA)}, name='Среда').to_frame())

,Среда
Python-библиотека Scikit-learn,1.8.0
Pandas,2.3.3
NumPy,2.4.6
Исходные таблицы,C:\Users\user\Downloads\BigData_Kit\course\data


**Что действительно записано в заказах**

Наличие статуса не означает наличие целевой переменной для любого вопроса о доставке. В исходной схеме нет фактического срока доставки и обещанного срока, поэтому задержку по ней восстановить нельзя.

In [2]:
# Шаг М2.01.02 — Что действительно записано в заказах
orders_source = pd.read_csv(COURSE_DATA / 'orders.csv')
customers_source = pd.read_csv(COURSE_DATA / 'customers.csv')
products_source = pd.read_csv(COURSE_DATA / 'products.csv')
display(pd.DataFrame({'Источник': ['Заказы', 'Покупатели', 'Товары'],
                      'Строк': [len(orders_source), len(customers_source), len(products_source)],
                      'Поля': [', '.join(x.columns) for x in [orders_source, customers_source, products_source]]}))
display(orders_source['status'].value_counts().to_frame('Заказов'))

,Источник,Строк,Поля
0,Заказы,200000,"order_id, order_ts, customer_id, city, product..."
1,Покупатели,20000,"customer_id, city, registered, segment"
2,Товары,2000,"product_id, category, name, price"


,Заказов
status,
delivered,164061
cancelled,17980
returned,10045
in_transit,7914


**Объект признаки и правильный ответ**

Признаки `X` доступны в момент решения. Цель `y` появляется позже. Классификация выбирает из заранее заданных классов, регрессия оценивает числовую величину. Несмотря на название, логистическая регрессия далее решает задачу классификации.

Классификация и регрессия отличаются смыслом ответа, а не форматом исходного CSV. Число 0/1 в бинарной разметке обозначает класс; его не следует интерпретировать как количество часов. Для обучения требуется известный правильный исход у исторических заявок, а для применения достаточно доступных входных характеристик.

Обобщение означает работу правила на наблюдениях, которые не использовались для настройки его параметров. Запоминание правильных ответов обучающих строк само по себе не подтверждает эту способность. Поэтому подготовка отдельной проверки является частью задачи до обучения, а не заключительным оформлением результата.

In [3]:
# Шаг М2.01.03 — Объект признаки и правильный ответ
task_contract = pd.DataFrame({
    'Понятие': ['Объект', 'Момент прогноза', 'Класс 1', 'Класс 0', 'Пример признака', 'Выход модели'],
    'Определение': ['Одна заявка', 'Оформление заявки', 'Доставка задержана',
                    'Задержки нет', 'Известная загрузка склада', 'Оценка риска класса 1']
})
display(task_contract)

,Понятие,Определение
0,Объект,Одна заявка
1,Момент прогноза,Оформление заявки
2,Класс 1,Доставка задержана
3,Класс 0,Задержки нет
4,Пример признака,Известная загрузка склада
5,Выход модели,Оценка риска класса 1


**Доступность данных до и после события**

Даже сильная связь с целью не делает столбец допустимым признаком. Причина исключения может быть временной, а не статистической: будущая компенсация слишком поздно появляется для прогноза при оформлении.

In [4]:
# Шаг М2.01.04 — Доступность данных до и после события
availability = pd.DataFrame({
    'Поле': ['distance_km', 'warehouse_load', 'weather_forecast', 'actual_delay_hours',
              'compensation_paid', 'customer_id'],
    'Когда известно': ['При оформлении', 'При оформлении', 'Прогноз при оформлении',
                       'После исхода', 'После исхода', 'При оформлении'],
    'Роль': ['Признак', 'Признак', 'Признак', 'Утечка будущего', 'Утечка будущего',
             'Идентификатор, исключён из X']
})
display(availability)

,Поле,Когда известно,Роль
0,distance_km,При оформлении,Признак
1,warehouse_load,При оформлении,Признак
2,weather_forecast,Прогноз при оформлении,Признак
3,actual_delay_hours,После исхода,Утечка будущего
4,compensation_paid,После исхода,Утечка будущего
5,customer_id,При оформлении,"Идентификатор, исключён из X"


Постановка фиксирует событие, момент решения и допустимые сведения. Отсутствующая целевая переменная не восстанавливается догадкой из похожего столбца.

## 2. Дополнительный источник с прозрачной генерацией

Учебный генератор использует фиксированное зерно случайности. Истинный риск зависит от нагрузки, расстояния, прогноза погоды и взаимодействия экспресс-доставки с высокой нагрузкой. Исход случаен даже при одинаковых признаках.

**Создание учебных заявок**

Названия городов и товарных категорий взяты из справочников курса; шкалы количества и суммы — из заказов. Сами заявки, сроки, нагрузка, погода и задержки созданы специально для практики. `order_amount` измеряется в условных денежных единицах.

После расчёта исхода добавляются пропуски в части входных измерений. В последнем квартале появляется новая категория города. Вероятность генератора не записывается в CSV и не передаётся обучаемым моделям.

Календарь является частью симуляции: итоговый учебный снимок относится к 8 января 2027 года, когда готовы метки всех заявок 2026 года. Это условная временная шкала, а не реальные наблюдения из будущего относительно даты занятия.

В генераторе гамма-распределение задаёт положительные расстояния с правым хвостом, бета-распределение — исходную шкалу нагрузки от нуля до единицы, а распределение Пуассона — целые неотрицательные количества предыдущих заказов. Это авторские настройки учебного источника. Бинарный исход разыгрывается как одно испытание с вероятностью задержки, рассчитанной по формуле; такие метки случайны даже при одинаковых признаках.

Начальная история заказов фиксируется для покупателя; к ней прибавляется число его более ранних заявок. Счётчик cumcount начинается с нуля, поэтому текущая заявка не включается в собственную историю.

In [5]:
# Шаг М2.01.05 — Создание учебных заявок
# Набор целиком учебный: справочники задают названия, генератор — зависимости.
rng = np.random.default_rng(42)
n = 14_000
source_customers = pd.read_csv(COURSE_DATA / 'customers.csv')
source_products = pd.read_csv(COURSE_DATA / 'products.csv')
source_orders = pd.read_csv(COURSE_DATA / 'orders.csv', usecols=['qty', 'amount'])
chosen_customers = source_customers.iloc[rng.integers(0, len(source_customers), n)].reset_index(drop=True)
chosen_products = source_products.iloc[rng.integers(0, len(source_products), n)].reset_index(drop=True)
times = pd.date_range('2026-01-01', '2026-12-31 23:59:00', periods=n)
distance = np.round(rng.gamma(2.1, 165, n).clip(5, 1_500), 1)
load = np.round(rng.beta(2.1, 2.8, n) * 1.5 + 0.1, 3)
delivery = rng.choice(['Обычная', 'Экспресс', 'Пункт выдачи'], n, p=[.55, .25, .20])
weather = rng.choice(['Ясно', 'Дождь', 'Снег'], n, p=[.64, .25, .11])
items = rng.choice(source_orders['qty'].to_numpy(), n)
amounts = rng.choice(source_orders['amount'].to_numpy(), n)
weekend = times.dayofweek >= 5
express = delivery == 'Экспресс'
snow = weather == 'Снег'
rain = weather == 'Дождь'
log_odds = (-4.6 + 1.45 * load + .0019 * distance + .40 * express
            + .65 * rain + 1.05 * snow + .32 * weekend
            + 1.35 * ((load > 1.0) & express)
            + .25 * (times.month >= 10) + .18 * (items >= 3))
true_probability = 1 / (1 + np.exp(-log_odds))
target = rng.binomial(1, true_probability)
late_hours = np.where(target == 1, rng.integers(1, 73, n), 0)
dataset = pd.DataFrame({
    'request_id': np.arange(1, n + 1), 'order_ts': times,
    'label_available_at': times + pd.Timedelta(days=7),
    'customer_id': chosen_customers['customer_id'],
    'city': chosen_customers['city'], 'product_category': chosen_products['category'],
    'distance_km': distance, 'items_count': items, 'order_amount': amounts,
    'warehouse_load': load, 'customer_prior_orders': rng.poisson(7, n),
    'promised_days': np.where(express, 1, np.where(delivery == 'Пункт выдачи', 4, 3)),
    'order_hour': times.hour, 'delivery_type': delivery, 'weather_forecast': weather,
    'is_weekend': np.where(weekend, 'Да', 'Нет'),
    'late_delivery': target, 'actual_delay_hours': late_hours,
    'compensation_paid': np.where(target == 1, rng.choice([0, 300, 500], n, p=[.25, .40, .35]), 0)
})
# Начальная история фиксирована для покупателя; текущая заявка в неё ещё не входит.
initial_history = dataset.groupby('customer_id')['customer_prior_orders'].transform('first')
dataset['customer_prior_orders'] = initial_history + dataset.groupby('customer_id', sort=False).cumcount()
# Пропуски в измерениях появляются после генерации исхода; это часть учебного источника.
for col in ['distance_km', 'warehouse_load', 'order_amount']:
    dataset.loc[rng.choice(n, int(n * .02), replace=False), col] = np.nan
for col in ['city', 'weather_forecast']:
    dataset.loc[rng.choice(n, int(n * .015), replace=False), col] = np.nan
late_indices = dataset.index[dataset['order_ts'].ge('2026-10-01')]
dataset.loc[rng.choice(late_indices, 80, replace=False), 'city'] = 'Новый город'
DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
dataset.to_csv(DATASET_PATH, index=False, date_format='%Y-%m-%d %H:%M:%S')
display(pd.Series({'Наблюдений': len(dataset), 'Доля задержек': dataset['late_delivery'].mean(),
                   'Числовых признаков': 7, 'Категориальных признаков': 5,
                   'Файл': str(DATASET_PATH)}, name='Учебный набор').to_frame())

,Учебный набор
Наблюдений,14000
Доля задержек,0.126143
Числовых признаков,7
Категориальных признаков,5
Файл,C:\Users\user\Downloads\BigData_Kit\course\pra...


**Проверка сохранённого учебного источника**

Повторное чтение проверяет именно тот файл, который получат остальные ноутбуки. Фиксированная частота заявок облегчает сравнение временных окон; покупатель может иметь несколько разных заявок.

In [6]:
# Шаг М2.01.06 — Проверка сохранённого учебного источника
data = pd.read_csv(DATASET_PATH, parse_dates=['order_ts', 'label_available_at'])
assert len(data) == 14_000 and data['request_id'].is_unique
assert data['late_delivery'].isin([0, 1]).all()
assert data['order_ts'].is_monotonic_increasing
assert data.groupby('customer_id')['customer_prior_orders'].diff().dropna().eq(1).all()
assert data['label_available_at'].sub(data['order_ts']).eq(pd.Timedelta(days=7)).all()
display(data.head(5))
display(data.dtypes.to_frame('Тип после чтения'))

,request_id,order_ts,label_available_at,customer_id,city,product_category,distance_km,items_count,order_amount,warehouse_load,customer_prior_orders,promised_days,order_hour,delivery_type,weather_forecast,is_weekend,late_delivery,actual_delay_hours,compensation_paid
0,1,2026-01-01 00:00:00,2026-01-08 00:00:00,1786,Нижний Новгород,Бытовая техника,116.6,1,3510.0,0.989,11,3,0,Обычная,Ясно,Нет,0,0,0
1,2,2026-01-01 00:37:32,2026-01-08 00:37:32,15480,Уфа,Красота,NaN,1,3740.0,1.053,9,3,0,Обычная,Ясно,Нет,0,0,0
2,3,2026-01-01 01:15:05,2026-01-08 01:15:05,13092,Омск,Дом и сад,365.4,1,1110.0,0.515,5,1,1,Экспресс,Ясно,Нет,0,0,0
3,4,2026-01-01 01:52:38,2026-01-08 01:52:38,8778,Пермь,Электроника,161.6,1,12270.0,0.673,7,4,1,Пункт выдачи,Дождь,Нет,0,0,0
4,5,2026-01-01 02:30:10,2026-01-08 02:30:10,8661,Москва,Бытовая техника,256.7,1,369500.0,0.870,4,1,2,Экспресс,Ясно,Нет,0,0,0


,Тип после чтения
request_id,int64
order_ts,datetime64[ns]
label_available_at,datetime64[ns]
customer_id,int64
city,object
product_category,object
distance_km,float64
items_count,int64
order_amount,float64
warehouse_load,float64


**Паспорт признаков и единицы измерения**

Числовые признаки имеют разные масштабы и единицы. Категории обозначают варианты, а не естественный числовой порядок. Допустимый список признаков фиксируется явно вместо удаления нескольких известных опасных полей.

Паспорт задаёт не только имя столбца, но и смысл измерения. `warehouse_load` представляет известную при оформлении относительную нагрузку и не является окончательной оценкой работы склада за весь день. `customer_prior_orders` относится только к более ранней истории. Подсчёт заказов клиента по полной будущей выгрузке под тем же названием создал бы другой, недопустимый признак.

Допустимость поля не устанавливается автоматически по его типу. И число, и строка могут содержать будущее. Проверка доступности требует знания процесса появления данных; анализ корреляций такую проверку не заменяет.

In [7]:
# Шаг М2.01.07 — Паспорт признаков и единицы измерения
NUMERIC = ['distance_km', 'items_count', 'order_amount', 'warehouse_load',
           'customer_prior_orders', 'promised_days', 'order_hour']
CATEGORICAL = ['city', 'product_category', 'delivery_type', 'weather_forecast', 'is_weekend']
FEATURES = NUMERIC + CATEGORICAL
TARGET = 'late_delivery'
feature_passport = pd.DataFrame({
    'Поле': FEATURES,
    'Содержание': ['Расстояние, км', 'Количество единиц', 'Сумма, усл. ед.',
                   'Относительная нагрузка', 'Предыдущие заказы', 'Обещанный срок, дни',
                   'Час оформления, 0–23', 'Город', 'Категория товара',
                   'Вид доставки', 'Прогноз погоды', 'Признак выходного']
})
assert not set(FEATURES) & {'actual_delay_hours', 'compensation_paid', TARGET, 'customer_id'}
display(feature_passport)

,Поле,Содержание
0,distance_km,"Расстояние, км"
1,items_count,Количество единиц
2,order_amount,"Сумма, усл. ед."
3,warehouse_load,Относительная нагрузка
4,customer_prior_orders,Предыдущие заказы
5,promised_days,"Обещанный срок, дни"
6,order_hour,"Час оформления, 0–23"
7,city,Город
8,product_category,Категория товара
9,delivery_type,Вид доставки


**Редкое событие и проверка полноты**

Дисбаланс классов означает неодинаковое число примеров разных классов. Он не исправляется простым удалением частого класса до разбиения. Отсутствие измерения отделяется от нулевого значения.

In [8]:
# Шаг М2.01.08 — Редкое событие и проверка полноты
display(data[TARGET].value_counts().rename_axis('Класс').to_frame('Заявок'))
display(data[FEATURES].isna().mean().mul(100).sort_values(ascending=False).to_frame('Пропуски, %'))
print('Заявок на покупателя в среднем:', round(len(data) / data['customer_id'].nunique(), 2))
print('Доля положительного класса:', round(data[TARGET].mean(), 4))

,Заявок
Класс,
0,12234
1,1766


,"Пропуски, %"
distance_km,2.0
order_amount,2.0
warehouse_load,2.0
city,1.5
weather_forecast,1.5
items_count,0.0
promised_days,0.0
customer_prior_orders,0.0
order_hour,0.0
product_category,0.0


Заявок на покупателя в среднем: 1.39
Доля положительного класса: 0.1261


**Почему будущий столбец даёт обманчивую точность**

Связь `actual_delay_hours > 0` с целевой переменной точна по определению генератора. Такой столбец полезен для контроля меток после события, но недоступен для раннего прогноза.

In [9]:
# Шаг М2.01.09 — Почему будущий столбец даёт обманчивую точность
leaked_answer = data['actual_delay_hours'].gt(0).astype(int)
display(pd.crosstab(data[TARGET], leaked_answer, rownames=['Правильный исход'],
                    colnames=['Ответ из будущего']))
print('Совпадение с целью:', leaked_answer.eq(data[TARGET]).mean())
print('Разрешён ли столбец в X:', 'actual_delay_hours' in FEATURES)

Ответ из будущего,0,1
Правильный исход,,
0,12234,0
1,0,1766


Совпадение с целью: 1.0
Разрешён ли столбец в X: False


Дополнительный источник сохранён с явными ролями полей и воспроизводимой случайностью. Будущие сведения присутствуют только для демонстрации утечки и исключены из X.

## 3. Временная проверка и зрелость меток

Обучающая часть определяет параметры модели. Валидационная помогает выбрать метод и правило решения. Тестовая используется после фиксации этих решений. Семидневные паузы между окнами нужны для поступления целевых меток.

**Разделение на обучение валидацию и тест**

Обучение заканчивается до 25 июля; валидация охватывает 1 августа — 23 сентября; тест начинается 1 октября. Промежуточные заявки не используются при выборе и проверке. Это осознанный временной зазор, а не потеря строк при очистке.

In [10]:
# Шаг М2.01.10 — Разделение на обучение валидацию и тест
train = data.loc[data['order_ts'].lt('2026-07-25')].copy()
valid = data.loc[data['order_ts'].ge('2026-08-01') & data['order_ts'].lt('2026-09-24')].copy()
test = data.loc[data['order_ts'].ge('2026-10-01')].copy()
split_summary = pd.DataFrame([
    {'Часть': name, 'Строк': len(frame), 'Начало': frame['order_ts'].min(),
     'Конец': frame['order_ts'].max()}
    for name, frame in [('Обучение', train), ('Валидация', valid), ('Тест', test)]
])
display(split_summary)
print('Строк в паузах:', len(data) - len(train) - len(valid) - len(test))

,Часть,Строк,Начало,Конец
0,Обучение,7863,2026-01-01 00:00:00,2026-07-24 23:42:27
1,Валидация,2072,2026-08-01 00:02:11,2026-09-23 23:58:51
2,Тест,3529,2026-10-01 00:18:35,2026-12-31 23:59:00


Строк в паузах: 536


**Проверяемые границы и отсутствие пересечения**

Контроль относится не только к дате заявки, но и к дате поступления метки. Тестовая часть остаётся закрытой для подбора модели; ниже используются только её границы и схема, а не метрики качества.

Валидация и тест различаются ролью в принятии решений. Выбор алгоритма, сложности и порога после просмотра валидации разрешён, но делает её результат зависимым от этого выбора. Окончательный тест нужен для оценки всей выбранной процедуры, а не для очередного раунда улучшения.

Временной зазор называют паузой или embargo. Его размер связан с готовностью разметки и возможным перекрытием событий, а не с удобным круглым числом строк. В учебном наборе доступность метки известна явно. В промышленной системе задержка её получения может быть неодинаковой для разных типов заявок.

In [11]:
# Шаг М2.01.11 — Проверяемые границы и отсутствие пересечения
assert train['label_available_at'].max() < valid['order_ts'].min()
assert valid['label_available_at'].max() < test['order_ts'].min()
assert set(train['request_id']).isdisjoint(valid['request_id'])
assert set(train['request_id']).isdisjoint(test['request_id'])
assert set(valid['request_id']).isdisjoint(test['request_id'])
print('Метки предыдущего окна готовы до начала следующего; заявки не пересекаются.')
display(train[TARGET].value_counts().to_frame('Обучающих примеров'))

Метки предыдущего окна готовы до начала следующего; заявки не пересекаются.


,Обучающих примеров
late_delivery,
0,6934
1,929


**Случайное разбиение отвечает на другой вопрос**

`train_test_split` полезен для подходящих независимых наблюдений. В данном временном сценарии случайное перемешивание смешивает прошлые и будущие заявки. Стратификация сохраняет доли классов, но сама не устраняет временную утечку.

In [12]:
# Шаг М2.01.12 — Случайное разбиение отвечает на другой вопрос
random_train, random_check = train_test_split(
    data[['request_id', 'order_ts', TARGET]], test_size=.25,
    stratify=data[TARGET], random_state=42)
display(pd.DataFrame({'Часть': ['Случайное обучение', 'Случайная проверка'],
                      'Минимальная дата': [random_train['order_ts'].min(), random_check['order_ts'].min()],
                      'Максимальная дата': [random_train['order_ts'].max(), random_check['order_ts'].max()]}))
print('В случайном обучении есть будущие октябрьские заявки:', random_train['order_ts'].ge('2026-10-01').any())

,Часть,Минимальная дата,Максимальная дата
0,Случайное обучение,2026-01-01 00:00:00,2026-12-31 23:59:00
1,Случайная проверка,2026-01-01 00:37:32,2026-12-31 23:21:27


В случайном обучении есть будущие октябрьские заявки: True


**Изменение состава данных во времени**

Различие распределений между окнами называется сдвигом данных. Здесь диагностируются доступные признаки и доля метки только в обучении и валидации. Новая категория теста будет обрабатываться заранее выбранным правилом, без обучения на тесте.

In [13]:
# Шаг М2.01.13 — Изменение состава данных во времени
display(pd.DataFrame({'Обучение': train[NUMERIC].median(), 'Валидация': valid[NUMERIC].median()}))
display(pd.crosstab(pd.concat([train.assign(split='Обучение'), valid.assign(split='Валидация')])['split'],
                    pd.concat([train, valid])['weather_forecast'], normalize='index').round(3))
print('Доли задержки train / valid:', round(train[TARGET].mean(), 3), round(valid[TARGET].mean(), 3))

,Обучение,Валидация
distance_km,296.450,288.850
items_count,1.000,1.000
order_amount,10120.000,10580.000
warehouse_load,0.718,0.734
customer_prior_orders,7.000,7.000
promised_days,3.000,3.000
order_hour,11.000,11.500


weather_forecast,Дождь,Снег,Ясно
split,,,
Валидация,0.253,0.112,0.635
Обучение,0.243,0.110,0.648


Доли задержки train / valid: 0.118 0.115


Части разделены по времени с учётом поступления ответов. Случайное разбиение показано для сопоставления и не подменяет основной эксперимент.

## 4. Обучаемые преобразования признаков

Правила заполнения и кодирования являются частью модели. Их состояние вычисляется только на обучении, а к другим частям применяется без повторного подбора. Числовая и категориальная ветви требуют разных действий.

**Матрица X и вектор y**

Строки X и y должны относиться к одним заявкам в одинаковом порядке. Разделение признаков и ответа делает последующие вызовы Scikit-learn явными. Метаданные времени остаются отдельно для проверки окон.

In [14]:
# Шаг М2.01.14 — Матрица X и вектор y
X_train, y_train = train[FEATURES], train[TARGET]
X_valid, y_valid = valid[FEATURES], valid[TARGET]
assert X_train.index.equals(y_train.index)
assert X_valid.index.equals(y_valid.index)
print('X_train:', X_train.shape, 'y_train:', y_train.shape)
display(X_train.head(3))

X_train: (7863, 12) y_train: (7863,)


,distance_km,items_count,order_amount,warehouse_load,customer_prior_orders,promised_days,order_hour,city,product_category,delivery_type,weather_forecast,is_weekend
0,116.6,1,3510.0,0.989,11,3,0,Нижний Новгород,Бытовая техника,Обычная,Ясно,Нет
1,NaN,1,3740.0,1.053,9,3,0,Уфа,Красота,Обычная,Ясно,Нет
2,365.4,1,1110.0,0.515,5,1,1,Омск,Дом и сад,Экспресс,Ясно,Нет


**Медианы только по обучающей части**

`SimpleImputer.fit` запоминает статистики обучения. `transform` применяет их. `fit_transform` объединяет оба действия и предназначен для той части, на которой разрешено учиться. Индикатор пропуска добавляет сведения об отсутствии измерения.

In [15]:
# Шаг М2.01.15 — Медианы только по обучающей части
numeric_imputer = SimpleImputer(strategy='median', add_indicator=True)
numeric_train = numeric_imputer.fit_transform(X_train[NUMERIC])
numeric_valid = numeric_imputer.transform(X_valid[NUMERIC])
display(pd.Series(numeric_imputer.statistics_, index=NUMERIC, name='Медиана train').to_frame())
print('Числовая ветвь после заполнения:', numeric_train.shape, numeric_valid.shape)
print('Столбцы с индикатором отсутствия:', [NUMERIC[i] for i in numeric_imputer.indicator_.features_])

,Медиана train
distance_km,296.450
items_count,1.000
order_amount,10120.000
warehouse_load,0.718
customer_prior_orders,7.000
promised_days,3.000
order_hour,11.000


Числовая ветвь после заполнения: (7863, 10) (2072, 10)
Столбцы с индикатором отсутствия: ['distance_km', 'order_amount', 'warehouse_load']


**Масштабирование числовых столбцов**

`StandardScaler` вычитает среднее обучения и делит на стандартное отклонение обучения. Единицы разных признаков перестают определять размер шага оптимизации. Нулевое среднее результата train не обязано сохраняться в более позднем окне.

Заполнение медианой меняет представление неизвестного значения, но не создаёт новое измерение. Индикатор отсутствия отделяет такую замену от настоящего значения, случайно равного медиане. При этом индикаторы создаются по столбцам, где отсутствие встретилось во время обучения; произвольное изменение их набора при применении нарушило бы схему.

Один подготовительный объект хранит разные состояния: статистики заполнения, параметры масштаба и словари вариантов. Эти состояния требуется сохранять вместе с моделью. Передача в применение только коэффициентов без подготовительных шагов даст другую функцию от исходной заявки.

In [16]:
# Шаг М2.01.16 — Масштабирование числовых столбцов
scaler = StandardScaler()
scaled_train = scaler.fit_transform(numeric_train)
scaled_valid = scaler.transform(numeric_valid)
display(pd.DataFrame({'Признак': NUMERIC,
                      'Среднее после преобразования': scaled_train[:, :len(NUMERIC)].mean(axis=0),
                      'Стандартное отклонение': scaled_train[:, :len(NUMERIC)].std(axis=0)}).round(4))
print('Заполнение не оставило NaN:', np.isfinite(scaled_valid).all())

,Признак,Среднее после преобразования,Стандартное отклонение
0,distance_km,-0.0,1.0
1,items_count,-0.0,1.0
2,order_amount,-0.0,1.0
3,warehouse_load,0.0,1.0
4,customer_prior_orders,0.0,1.0
5,promised_days,0.0,1.0
6,order_hour,-0.0,1.0


Заполнение не оставило NaN: True


**Категории как отдельные индикаторы**

One-hot encoding создаёт столбец для каждого значения категории. Вектор не вводит произвольный порядок городов. Список известных категорий также должен быть выучен только на train.

In [17]:
# Шаг М2.01.17 — Категории как отдельные индикаторы
category_imputer = SimpleImputer(strategy='most_frequent')
category_train = category_imputer.fit_transform(X_train[CATEGORICAL])
category_valid = category_imputer.transform(X_valid[CATEGORICAL])
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_train = encoder.fit_transform(category_train)
encoded_valid = encoder.transform(category_valid)
encoded_names = encoder.get_feature_names_out(CATEGORICAL)
display(pd.DataFrame(encoded_train[:4], columns=encoded_names).iloc[:, :10])
print('Категориальных индикаторов:', len(encoded_names))

,city_Волгоград,city_Воронеж,city_Екатеринбург,city_Казань,city_Краснодар,city_Красноярск,city_Москва,city_Нижний Новгород,city_Новосибирск,city_Омск
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Категориальных индикаторов: 33


**Неизвестный вариант не должен менять схему**

Заранее установленное `handle_unknown='ignore'` сохраняет число столбцов при новом значении. Новая категория получает нули во всём блоке соответствующего исходного признака. Это техническая совместимость, а не гарантия точного прогноза.

In [18]:
# Шаг М2.01.18 — Неизвестный вариант не должен менять схему
new_example = X_valid[CATEGORICAL].head(1).copy()
new_example.loc[:, 'city'] = 'Город, отсутствующий в train'
new_encoded = encoder.transform(category_imputer.transform(new_example))
city_columns = np.array([name.startswith('city_') for name in encoded_names])
print('Сумма индикаторов города:', new_encoded[:, city_columns].sum())
print('Ширина представления сохранена:', new_encoded.shape[1] == encoded_train.shape[1])

Сумма индикаторов города: 0.0
Ширина представления сохранена: True


Заполнение, масштабирование и кодирование обучены на train и применены к validation без повторного обучения. Для разных типов признаков получены совместимые числовые представления.

## 5. Единая процедура подготовки и контроль договора

Pipeline связывает шаги последовательно, а ColumnTransformer выполняет разные ветви над выбранными столбцами. Вместе они позволяют передать исходную таблицу в будущую модель без ручного повторения преобразований.

**Числовая и категориальная ветви**

Каждый шаг Pipeline получает выход предыдущего. Имена шагов нужны для доступа к состоянию и настройки параметров. Числовая ветвь будет использоваться для логистической регрессии; дерево и лес сохранят заполнение, но обойдутся без StandardScaler.

In [19]:
# Шаг М2.01.19 — Числовая и категориальная ветви
numeric_branch = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True)),
    ('scale', StandardScaler())
])
category_branch = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
display(pd.DataFrame({'Ветвь': ['Числовая', 'Категориальная'],
                      'Шаги': ['Заполнение → масштабирование', 'Заполнение → индикаторы']}))

,Ветвь,Шаги
0,Числовая,Заполнение → масштабирование
1,Категориальная,Заполнение → индикаторы


**ColumnTransformer на разрешённом наборе**

Ветви получают столбцы по именам. Остальные поля удаляются через `remainder='drop'`. Выходы ветвей соединяются в одном порядке; имена позволяют проверить, каким исходным признакам соответствуют числовые позиции.

На входе остаются двенадцать содержательных признаков, но на выходе может быть больше числовых столбцов. Количество вариантов города и категории определяет ширину one-hot блока, а индикаторы пропусков добавляют ещё несколько столбцов. Поэтому число коэффициентов модели должно сопоставляться с именами выхода преобразователя.

Выбор столбцов по именам устойчив к перестановке исходной таблицы. Для уже полученного массива NumPy имён нет, и соответствие задаётся позициями. Смешивание этих соглашений без проверки может сохранить правильные размеры и одновременно подставить не те значения под коэффициенты.

In [20]:
# Шаг М2.01.20 — ColumnTransformer на разрешённом наборе
prepare = ColumnTransformer([
    ('numeric', numeric_branch, NUMERIC),
    ('categorical', category_branch, CATEGORICAL)
], remainder='drop', verbose_feature_names_out=True)
Z_train = prepare.fit_transform(X_train)
Z_valid = prepare.transform(X_valid)
names = prepare.get_feature_names_out()
display(pd.DataFrame(Z_train[:5], columns=names).iloc[:, :12].round(3))
print('Вход / числовой выход:', X_train.shape, Z_train.shape)
assert Z_train.shape[1] == Z_valid.shape[1] == len(names)

,numeric__distance_km,numeric__items_count,numeric__order_amount,numeric__warehouse_load,numeric__customer_prior_orders,numeric__promised_days,numeric__order_hour,numeric__missingindicator_distance_km,numeric__missingindicator_order_amount,numeric__missingindicator_warehouse_load,categorical__city_Волгоград,categorical__city_Воронеж
0,-0.970,-0.544,-0.514,0.837,1.420,0.286,-1.661,-0.138,-0.143,-0.144,0.0,0.0
1,-0.217,-0.544,-0.510,1.050,0.678,0.286,-1.661,7.245,-0.143,-0.144,0.0,0.0
2,0.072,-0.544,-0.560,-0.741,-0.806,-1.604,-1.517,-0.138,-0.143,-0.144,0.0,0.0
3,-0.782,-0.544,-0.350,-0.215,-0.064,1.231,-1.517,-0.138,-0.143,-0.144,0.0,0.0
4,-0.383,-0.544,6.363,0.441,-1.177,-1.604,-1.372,-0.138,-0.143,-0.144,0.0,0.0


Вход / числовой выход: (7863, 12) (7863, 43)


**Состояние обученной подготовки**

Доступ через `named_transformers_` и `named_steps` позволяет проверить выученное состояние. Это диагностика процедуры, а не новый расчёт статистик по валидации.

In [21]:
# Шаг М2.01.21 — Состояние обученной подготовки
fitted_numeric = prepare.named_transformers_['numeric']
display(pd.Series(fitted_numeric.named_steps['impute'].statistics_, index=NUMERIC,
                   name='Сохранённые медианы').to_frame())
print('Все числа конечны:', np.isfinite(Z_train).all(), np.isfinite(Z_valid).all())
print('Выученные варианты типа доставки:',
      prepare.named_transformers_['categorical'].named_steps['encode'].categories_[2])

,Сохранённые медианы
distance_km,296.450
items_count,1.000
order_amount,10120.000
warehouse_load,0.718
customer_prior_orders,7.000
promised_days,3.000
order_hour,11.000


Все числа конечны: True True
Выученные варианты типа доставки: ['Обычная' 'Пункт выдачи' 'Экспресс']


**Одна заявка и пакет получают один результат**

Преобразование новых объектов не должно зависеть от того, пришли они по одному или в составе пакета. Обученное состояние остаётся прежним.

In [22]:
# Шаг М2.01.22 — Одна заявка и пакет получают один результат
one_at_a_time = np.vstack([prepare.transform(X_valid.iloc[[i]]) for i in range(5)])
together = prepare.transform(X_valid.iloc[:5])
assert np.allclose(one_at_a_time, together)
assert np.array_equal(fitted_numeric.named_steps['impute'].statistics_, numeric_imputer.statistics_)
print('Состав пакета не меняет подготовку одной и той же заявки.')

Состав пакета не меняет подготовку одной и той же заявки.


**Временные окна внутри обучения**

При будущем подборе параметров понадобится повторная проверка только внутри train. TimeSeriesSplit сохраняет направление времени. `gap` измеряется числом строк, поэтому соответствие семи дням проверяется отдельно.

In [23]:
# Шаг М2.01.23 — Временные окна внутри обучения
cv = TimeSeriesSplit(n_splits=3, gap=300)
folds = []
for fold, (fit_idx, check_idx) in enumerate(cv.split(X_train), 1):
    fit_part, check_part = train.iloc[fit_idx], train.iloc[check_idx]
    assert fit_part['label_available_at'].max() < check_part['order_ts'].min()
    folds.append({'Окно': fold, 'Обучение': len(fit_idx), 'Проверка': len(check_idx),
                  'Последняя готовая метка': fit_part['label_available_at'].max(),
                  'Начало проверки': check_part['order_ts'].min()})
display(pd.DataFrame(folds))

,Окно,Обучение,Проверка,Последняя готовая метка,Начало проверки
0,1,1668,1965,2026-02-20 11:08:17,2026-02-21 07:29:28
1,2,3633,1965,2026-04-12 16:45:08,2026-04-13 13:06:19
2,3,5598,1965,2026-06-02 22:21:58,2026-06-03 18:43:10


**Контрольная задача подготовки новых заявок**

Небольшой пакет содержит пропуск нагрузки и неизвестный прогноз погоды. Правило подготовки уже определено; новое обучение не выполняется. Целевой столбец при применении не требуется.

Проверенный преобразователь решает техническую задачу согласованного представления. Его успешная работа на неизвестном городе не означает, что модель получила исторические примеры этого города. При изменении условий требуется отдельно контролировать частоту неизвестных вариантов и качество, когда поступят правильные метки.

В дальнейших ноутбуках исходная таблица проходит подготовку внутри Pipeline при каждом предсказании. Повторное обучение подготовки на новых пакетах не выполняется. Эта граница предотвращает зависимость одной заявки от произвольного состава соседних строк.

In [24]:
# Шаг М2.01.24 — Контрольная задача подготовки новых заявок
new_requests = X_valid.head(3).copy()
new_requests.iloc[0, new_requests.columns.get_loc('warehouse_load')] = np.nan
new_requests.iloc[1, new_requests.columns.get_loc('weather_forecast')] = 'Туман'
ready_requests = prepare.transform(new_requests)
display(new_requests[['distance_km', 'warehouse_load', 'weather_forecast']])
print('Готовый пакет:', ready_requests.shape, 'конечные значения:', np.isfinite(ready_requests).all())
assert TARGET not in new_requests.columns
assert ready_requests.shape[1] == Z_train.shape[1]

,distance_km,warehouse_load,weather_forecast
8131,18.1,NaN,Ясно
8132,592.7,0.705,Туман
8133,430.6,0.601,Ясно


Готовый пакет: (3, 43) конечные значения: True


Получен единый подготовительный объект с обученным состоянием и проверенной схемой выхода. Он станет первым шагом Pipeline с классификатором.

## Полученный результат

Сохранён дополнительный учебный источник, определён разрешённый набор признаков и сформированы временные части с паузами на созревание меток. Подготовка обучена только на train и проверена на валидации и новых заявках.

Дальше одинаковый договор данных используется при изучении логистической регрессии, дерева, случайного леса и конечного выбора решения. Значения будущих исходов и тестовые метрики не участвовали в подготовке.

## Документация

[Подготовка и утечка данных](https://scikit-learn.org/1.8/common_pitfalls.html)

[Композиция преобразований](https://scikit-learn.org/1.8/modules/compose.html)